# V3 SAC Training for the ETS Gymnasium Environment

This notebook trains a Soft Actor-Critic (SAC) agent on the V2 ETS Gymnasium environment.

## Workflow

1. Configure the EnergyPlus CSV path and training parameters.
2. Create separate training and evaluation environments.
3. Train the SAC agent.
4. Save checkpoints and the best-performing model.
5. Evaluate the final policy on the held-out test period.
6. Generate performance metrics and detailed test results.

The dataset is split chronologically to reduce data leakage between
training and testing.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.vec_env import DummyVecEnv

from ets_gym_v2 import ETSGymV2

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

## Configuration

Set the training parameters below:

In [2]:
# =====================================================
# CONFIGURATION VARIABLES
# =====================================================

# Path to the EnergyPlus output CSV file
input_csv = "/Users/ashanperera/EnergyPlus/RL_UAE_HVAC_SOFTWARE/eplusout.csv"

# Number of training timesteps
timesteps = 100_000

# Fraction of data to use for training (0.8 = 80% train, 20% test)
train_fraction = 0.8

# Random seed for reproducibility
seed = 42

print("Configuration:")
print(f"  Input CSV: {input_csv}")
print(f"  Training timesteps: {timesteps:,}")
print(f"  Train/test split: {train_fraction:.0%} / {1-train_fraction:.0%}")
print(f"  Seed: {seed}")

Configuration:
  Input CSV: /Users/ashanperera/EnergyPlus/RL_UAE_HVAC_SOFTWARE/eplusout.csv
  Training timesteps: 100,000
  Train/test split: 80% / 20%
  Seed: 42


## Environment Factory Functions

These functions create training and evaluation environments with proper episode boundaries:
- **make_train_env**: Creates environments restricted to training data period
- **make_eval_env**: Creates environments for test period evaluation

In [3]:
def make_train_env(csv_path: str, split_index: int):
    def _init():
        env = ETSGymV2(csv_path, random_start=True)
        # Restrict random starts to training region.
        original_reset = env.reset

        def reset_train(*, seed=None, options=None):
            if options is None:
                options = {}
            max_start = max(0, split_index - env.episode_steps - 1)
            if "start_index" not in options:
                if max_start > 0:
                    start = int(env.np_random.integers(0, max_start + 1))
                else:
                    start = 0
                options["start_index"] = start
            return original_reset(seed=seed, options=options)

        env.reset = reset_train
        return Monitor(env)

    return _init

In [4]:
def make_eval_env(csv_path: str, split_index: int):
    def _init():
        env = ETSGymV2(csv_path, random_start=False)
        original_reset = env.reset

        # Create several fixed evaluation start points across the test period
        max_start = len(env.df) - env.episode_steps - 1

        eval_starts = np.linspace(
            split_index,
            max_start,
            num=5,
            dtype=int
        )

        eval_counter = {"i": 0}

        def reset_eval(*, seed=None, options=None):
            if options is None:
                options = {}

            start = int(
                eval_starts[
                    eval_counter["i"] % len(eval_starts)
                ]
            )

            eval_counter["i"] += 1

            options["start_index"] = start

            return original_reset(
                seed=seed,
                options=options
            )

        env.reset = reset_eval

        return Monitor(env)

    return _init

## Policy Evaluation Function

Evaluates the trained model on the full test period and generates results CSV with metrics:
- Cooling delivery metrics (required, delivered, unmet)
- Pump energy consumption
- Temperature differential statistics
- Diagnostic low-ΔT occurrence based on the current 4°C EnergyPlus design benchmark

> Note: 4°C is used here as a design/engineering diagnostic benchmark and
> should not be interpreted as a confirmed regulatory low-ΔT threshold.

In [5]:
def evaluate_policy_full_test(
    model,
    csv_path: str,
    split_index: int,
    output_csv: Path
):
    env = ETSGymV2(csv_path, random_start=False )
    rows = []
    start = split_index

    # We deliberately stop one row before the absolute end because
    # ETSGymV2 constructs the next observation after each step.
    final_step_index = len(env.df) - 2
    total_steps = final_step_index - start + 1

    obs, info = env.reset(
        options={"start_index": start}
    )

    env.episode_steps = total_steps
    total_reward = 0.0
    required_wh = 0.0
    delivered_wh = 0.0
    unmet_wh = 0.0
    pump_wh = 0.0
    done = False

    while not done:

        action, _ = model.predict(
            obs,
            deterministic=True
        )

        obs, reward, terminated, truncated, step_info = env.step(action)

        required_wh += step_info["q_required_w"] * 0.25
        delivered_wh += step_info["q_delivered_w"] * 0.25
        unmet_wh += step_info["q_unmet_w"] * 0.25
        pump_wh += step_info["pump_power_w"] * 0.25

        total_reward += reward

        rows.append({
            "dataset_index":
                step_info["dataset_index"],

            "reward":
                reward,

            "flow_command":
                step_info["flow_command"],

            "flow_kg_s":
                step_info["flow_kg_s"],

            "q_required_w":
                step_info["q_required_w"],

            "q_delivered_w":
                step_info["q_delivered_w"],

            "q_unmet_w":
                step_info["q_unmet_w"],

            "delta_t_c":
                step_info["delta_t_c"],

            "pump_power_w":
                step_info["pump_power_w"],

            "low_delta_t":
                step_info["low_delta_t"],

            "low_delta_t_severity":
                step_info["low_delta_t_severity"],
        })

        done = terminated or truncated

    result_df = pd.DataFrame(rows)

    result_df.to_csv(
        output_csv,
        index=False
    )

    active = result_df[
        result_df["q_required_w"] > 10.0
    ]

    metrics = {

        "test_start_index":
            int(split_index),

        "test_steps":
            int(len(result_df)),

        "total_reward":
            float(total_reward),

        "required_cooling_kwh_th":
            float(required_wh / 1000.0),

        "delivered_cooling_kwh_th":
            float(delivered_wh / 1000.0),

        "unmet_cooling_kwh_th":
            float(unmet_wh / 1000.0),

        "delivery_fraction_pct":
            float(
                100.0 * delivered_wh / required_wh
                if required_wh > 0
                else 100.0
            ),

        "pump_energy_kwh":
            float(pump_wh / 1000.0),

        "mean_delta_t_c_active":
            float(
                active["delta_t_c"].mean()
            ) if len(active) else 0.0,

        "median_delta_t_c_active":
            float(
                active["delta_t_c"].median()
            ) if len(active) else 0.0,

        "mean_flow_command_active":
            float(
                active["flow_command"].mean()
            ) if len(active) else 0.0,

        "low_delta_t_fraction_pct_active":
            float(
                100.0 *
                active["low_delta_t"].mean()
            ) if len(active) else 0.0,
    }

    return metrics

## Main Training Function

Sets up SAC training with:

- notebook configuration variables for input CSV, training timesteps,
  train/test fraction and random seed
- chronological train/test splitting aligned to 24-hour episodes
- vectorised training and evaluation environments
- checkpoint and validation callbacks
- SAC neural-network configuration
- saving of the final and best-performing models
- held-out test evaluation and metric generation

In [6]:
def main(input_csv, timesteps, train_fraction, seed):
    """Train SAC agent on ETS environment."""
    csv_path = str(Path(input_csv).resolve())
    df = pd.read_csv(csv_path)

    split_index = int(len(df) * train_fraction)

    # Align split to whole 24-hour episodes.
    split_index = (split_index // 96) * 96

    output_dir = Path("v3_sac_output")
    output_dir.mkdir(exist_ok=True)

    train_env = DummyVecEnv([make_train_env(csv_path, split_index)])
    eval_env = DummyVecEnv([make_eval_env(csv_path, split_index)])

    checkpoint_callback = CheckpointCallback(
        save_freq=10_000,
        save_path=str(output_dir / "checkpoints"),
        name_prefix="sac_ets"
    )

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(output_dir / "best_model"),
        log_path=str(output_dir / "eval_logs"),
        eval_freq=5_000,
        deterministic=True,
        render=False,
        n_eval_episodes=5,
    )

    model = SAC(
        "MlpPolicy",
        train_env,
        learning_rate=3e-4,
        buffer_size=100_000,
        learning_starts=2_000,
        batch_size=256,
        tau=0.005,
        gamma=0.99,
        train_freq=1,
        gradient_steps=1,
        ent_coef="auto",
        verbose=1,
        seed=seed,
        tensorboard_log=str(output_dir / "tensorboard"),
        policy_kwargs=dict(net_arch=[256, 256]),
    )

    print("\nV3 SAC training")
    print("=" * 55)
    print(f"Dataset rows          : {len(df)}")
    print(f"Training rows         : {split_index}")
    print(f"Testing rows          : {len(df) - split_index}")
    print(f"Training fraction     : {train_fraction:.2f}")
    print(f"Training timesteps    : {timesteps}")
    print(f"Episode length        : 96 steps (24 h)")
    print()

    model.learn(
        total_timesteps=timesteps,
        callback=[checkpoint_callback, eval_callback],
        progress_bar=True,
    )

    final_model_path = output_dir / "sac_ets_v3_final"
    model.save(str(final_model_path))

    best_model_path = (
        output_dir /
        "best_model" /
        "best_model.zip"
    )

    if best_model_path.exists():
        best_model = SAC.load(
            str(best_model_path)
        )
        print(f"Loaded best model: {best_model_path}")
    else:
        print(
            "Best model not found. "
            "Using final model instead."
        )
        best_model = model

    evaluation_csv = output_dir / "sac_test_results.csv"

    final_metrics = evaluate_policy_full_test(
        model,
        csv_path,
        split_index,
        output_dir / "sac_final_test_results.csv"
    )

    best_metrics = evaluate_policy_full_test(
        best_model,
        csv_path,
        split_index,
        output_dir / "sac_best_test_results.csv"
    )

    metrics = evaluate_policy_full_test(
        best_model,
        csv_path,
        split_index,
        evaluation_csv
    )

    metrics["training_rows"] = split_index
    metrics["testing_rows"] = len(df) - split_index
    metrics["training_timesteps"] = timesteps
    metrics["seed"] = seed
    metrics["testing_transitions"] = metrics["test_steps"]

    metrics_path = output_dir / "sac_test_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2))

    print("\nV3 training complete")
    print("=" * 55)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k:38s}: {v:.4f}")
        else:
            print(f"{k:38s}: {v}")

    print(f"\nFinal model     : {final_model_path}.zip")
    print(f"Test results    : {evaluation_csv}")
    print(f"Test metrics    : {metrics_path}")
    print(f"Best model dir  : {output_dir / 'best_model'}")
    print(f"Testing observations   : {len(df) - split_index}")
    print(f"Testing transitions    : {metrics['test_steps']}")
    
    return model, metrics

In [7]:
# =====================================================
# RUN TRAINING
# =====================================================
# Execute this cell to start training with the configuration above

model, metrics = main(input_csv, timesteps, train_fraction, seed)

Using cpu device

V3 SAC training
Dataset rows          : 35232
Training rows         : 28128
Testing rows          : 7104
Training fraction     : 0.80
Training timesteps    : 100000
Episode length        : 96 steps (24 h)

Logging to v3_sac_output/tensorboard/SAC_8
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96       |
|    ep_rew_mean     | -242     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 8165     |
|    time_elapsed    | 0        |
|    total_timesteps | 384      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96       |
|    ep_rew_mean     | -244     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 9633     |
|    time_elapsed    | 0        |
|    total_timesteps | 768      |
---------------------------------
---------------------------------
| rollout/         

Eval num_timesteps=5000, episode_reward=-185.60 +/- 15.33

Episode length: 96.00 +/- 0.00

---------------------------------
| eval/              |          |
|    mean_ep_length  | 96       |
|    mean_reward     | -186     |
| time/              |          |
|    total_timesteps | 5000     |
| train/             |          |
|    actor_loss      | 34.3     |
|    critic_loss     | 2.61     |
|    ent_coef        | 0.474    |
|    ent_coef_loss   | -0.522   |
|    learning_rate   | 0.0003   |
|    n_updates       | 2999     |
---------------------------------


New best mean reward!

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96       |
|    ep_rew_mean     | -217     |
| time/              |          |
|    episodes        | 56       |
|    fps             | 604      |
|    time_elapsed    | 8        |
|    total_timesteps | 5376     |
| train/             |          |
|    actor_loss      | 37.3     |
|    critic_loss     | 3.76     |
|    ent_coef        | 0.43     |
|    ent_coef_loss   | -0.962   |
|    learning_rate   | 0.0003   |
|    n_updates       | 3375     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96       |
|    ep_rew_mean     | -212     |
| time/              |          |
|    episodes        | 60       |
|    fps             | 582      |
|    time_elapsed    | 9        |
|    total_timesteps | 5760     |
| train/             |          |
|    actor_loss      | 39       |
|    critic_loss     | 2.86     |
|    ent_coef 

... (74 outputs left out)

Eval num_timesteps=100000, episode_reward=-60.72 +/- 2.34

Episode length: 96.00 +/- 0.00

---------------------------------
| eval/              |          |
|    mean_ep_length  | 96       |
|    mean_reward     | -60.7    |
| time/              |          |
|    total_timesteps | 100000   |
| train/             |          |
|    actor_loss      | 115      |
|    critic_loss     | 0.454    |
|    ent_coef        | 0.184    |
|    ent_coef_loss   | -0.0779  |
|    learning_rate   | 0.0003   |
|    n_updates       | 97999    |
---------------------------------


Loaded best model: v3_sac_output/best_model/best_model.zip

V3 training complete
test_start_index                      : 28128
test_steps                            : 7103
total_reward                          : -5100.9144
required_cooling_kwh_th               : 5690.0845
delivered_cooling_kwh_th              : 5687.7928
unmet_cooling_kwh_th                  : 2.2917
delivery_fraction_pct                 : 99.9597
pump_energy_kwh                       : 112.9439
mean_delta_t_c_active                 : 3.0302
median_delta_t_c_active               : 3.3119
mean_flow_command_active              : 0.2848
low_delta_t_fraction_pct_active       : 98.9488
training_rows                         : 28128
testing_rows                          : 7104
training_timesteps                    : 100000
seed                                  : 42
testing_transitions                   : 7103

Final model     : v3_sac_output/sac_ets_v3_final.zip
Test results    : v3_sac_output/sac_test_results.csv
Test metric

## Results Summary

Explore the training results above. Key outputs are saved in `v3_sac_output/`:
- **Best model**: `best_model/` - the best model checkpoint
- **Final model**: `sac_ets_v3_final.zip` - final trained model
- **Test results**: `sac_test_results.csv` - detailed step-by-step results
- **Metrics**: `sac_test_metrics.json` - aggregated performance metrics
- **TensorBoard logs**: `tensorboard/` - training curves and diagnostics

In [8]:
# Display metrics as a summary table
if 'metrics' in locals():
    print("\n" + "="*60)
    print("TRAINING METRICS SUMMARY")
    print("="*60)
    metrics_df = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])
    print(metrics_df.to_string(index=False))
    print("="*60)


TRAINING METRICS SUMMARY
                         Metric         Value
               test_start_index  28128.000000
                     test_steps   7103.000000
                   total_reward  -5100.914428
        required_cooling_kwh_th   5690.084457
       delivered_cooling_kwh_th   5687.792791
           unmet_cooling_kwh_th      2.291666
          delivery_fraction_pct     99.959725
                pump_energy_kwh    112.943898
          mean_delta_t_c_active      3.030161
        median_delta_t_c_active      3.311896
       mean_flow_command_active      0.284828
low_delta_t_fraction_pct_active     98.948833
                  training_rows  28128.000000
                   testing_rows   7104.000000
             training_timesteps 100000.000000
                           seed     42.000000
            testing_transitions   7103.000000
